In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
anime_path = Path(r"..\data\raw\anime")

anime = pd.read_csv(anime_path / "animes.csv")
ratings = pd.read_csv(anime_path / "profiles.csv")


In [3]:
anime.head()

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,img_url,link
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,https://cdn.myanimelist.net/images/anime/9/766...,https://myanimelist.net/anime/28891/Haikyuu_Se...
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,https://cdn.myanimelist.net/images/anime/3/671...,https://myanimelist.net/anime/23273/Shigatsu_w...
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,https://cdn.myanimelist.net/images/anime/6/867...,https://myanimelist.net/anime/34599/Made_in_Abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,https://cdn.myanimelist.net/images/anime/1223/...,https://myanimelist.net/anime/5114/Fullmetal_A...
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,https://cdn.myanimelist.net/images/anime/3/815...,https://myanimelist.net/anime/31758/Kizumonoga...


In [4]:
ratings.head()

,profile,gender,birthday,favorites_anime,link
0,DesolatePsyche,Male,"Oct 2, 1994","['33352', '25013', '5530', '33674', '1482', '2...",https://myanimelist.net/profile/DesolatePsyche
1,baekbeans,Female,"Nov 10, 2000","['11061', '31964', '853', '20583', '918', '925...",https://myanimelist.net/profile/baekbeans
2,skrn,NaN,NaN,"['918', '2904', '11741', '17074', '23273', '32...",https://myanimelist.net/profile/skrn
3,edgewalker00,Male,Sep 5,"['5680', '849', '2904', '3588', '37349']",https://myanimelist.net/profile/edgewalker00
4,aManOfCulture99,Male,"Oct 30, 1999","['4181', '7791', '9617', '5680', '2167', '4382...",https://myanimelist.net/profile/aManOfCulture99


In [5]:
from pprint import pprint

pprint("Anime head:")
pprint(anime.head())

pprint("\nAnime info:")
anime.info()

pprint("\nAnime shape:")
pprint(anime.shape)

pprint("\nAnime null values:")
pprint(anime.isna().sum())

pprint("\nRatings head:")
pprint(ratings.head())

pprint("\nRatings info:")
ratings.info()

pprint("\nRatings shape:")
pprint(ratings.shape)

pprint("\nRatings null values:")
pprint(ratings.isna().sum())

'Anime head:'
     uid                             title  \
0  28891           Haikyuu!! Second Season   
1  23273           Shigatsu wa Kimi no Uso   
2  34599                     Made in Abyss   
3   5114  Fullmetal Alchemist: Brotherhood   
4  31758  Kizumonogatari III: Reiketsu-hen   

                                            synopsis  \
0  Following their participation at the Inter-Hig...   
1  Music accompanies the path of the human metron...   
2  The Abyss—a gaping chasm stretching down into ...   
3  "In order for something to be obtained, someth...   
4  After helping revive the legendary vampire Kis...   

                                               genre  \
0  ['Comedy', 'Sports', 'Drama', 'School', 'Shoun...   
1  ['Drama', 'Music', 'Romance', 'School', 'Shoun...   
2  ['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...   
3  ['Action', 'Military', 'Adventure', 'Comedy', ...   
4   ['Action', 'Mystery', 'Supernatural', 'Vampire']   

                          aired  ep

In [6]:
import sys
from pathlib import Path

src_file_path = Path.cwd().parent
sys.path.insert(0,str(src_file_path))
from src.text_normalizer import Text_Normalization


norm = Text_Normalization()

import ast
from pathlib import Path
import pandas as pd

# -----------------------------
# 1. Clean genre lists
# -----------------------------

def clean_genre(value):
    if pd.isna(value):
        return ""

    try:
        genres = ast.literal_eval(value)
        return " ".join(genres)
    except:
        return str(value)


# -----------------------------
# 2. Remove anime without synopsis
# -----------------------------

anime = anime.dropna(subset=["synopsis"]).copy()


# -----------------------------
# 3. Clean genres
# -----------------------------

anime["genre"] = anime["genre"].apply(clean_genre)


# -----------------------------
# 4. Create semantic tag
# -----------------------------

anime["tag"] = (
    anime["title"].fillna("") + " " +
    anime["genre"].fillna("") + " " +
    anime["synopsis"].fillna("")
).str.lower()


# -----------------------------
# 5. Normalize titles
# -----------------------------

anime["title_normalized"] = anime["title"].apply(norm.normalize)


# -----------------------------
# 6. Keep useful columns
# -----------------------------

anime = anime[
    [
        "uid",
        "title",
        "synopsis",
        "genre",
        "aired",
        "episodes",
        "members",
        "popularity",
        "ranked",
        "score",
        "tag",
        "title_normalized",
        "img_url",
        "link"
    ]
].copy()


# -----------------------------
# 7. Check result
# -----------------------------

print("Shape:", anime.shape)
print("Missing synopsis:", anime["synopsis"].isna().sum())
print("Missing genre:", anime["genre"].isna().sum())
print("Missing tag:", anime["tag"].isna().sum())
print("Missing normalized title:", anime["title_normalized"].isna().sum())


# -----------------------------
# 8. Save processed dataset
# -----------------------------

processed_path = Path("../data/processed")

anime.to_pickle(
    processed_path / "anime.pkl"
)

print("Saved:", processed_path / "anime.pkl")

Shape: (18336, 14)
Missing synopsis: 0
Missing genre: 0
Missing tag: 0
Missing normalized title: 0
Saved: ..\data\processed\anime.pkl


In [7]:
myanime = pd.read_pickle(r"..\data\processed\anime.pkl")

In [8]:
myanime.head()

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,tag,title_normalized,img_url,link
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,Comedy Sports Drama School Shounen,"Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,haikyuu!! second season comedy sports drama sc...,haikyuu second season,https://cdn.myanimelist.net/images/anime/9/766...,https://myanimelist.net/anime/28891/Haikyuu_Se...
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,Drama Music Romance School Shounen,"Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,shigatsu wa kimi no uso drama music romance sc...,shigatsu wa kimi no uso,https://cdn.myanimelist.net/images/anime/3/671...,https://myanimelist.net/anime/23273/Shigatsu_w...
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,Sci-Fi Adventure Mystery Drama Fantasy,"Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,made in abyss sci-fi adventure mystery drama f...,made in abyss,https://cdn.myanimelist.net/images/anime/6/867...,https://myanimelist.net/anime/34599/Made_in_Abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...",Action Military Adventure Comedy Drama Magic F...,"Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,fullmetal alchemist: brotherhood action milita...,fullmetal alchemist brotherhood,https://cdn.myanimelist.net/images/anime/1223/...,https://myanimelist.net/anime/5114/Fullmetal_A...
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,Action Mystery Supernatural Vampire,"Jan 6, 2017",1.0,214621,502,22.0,8.83,kizumonogatari iii: reiketsu-hen action myster...,kizumonogatari iii reiketsuhen,https://cdn.myanimelist.net/images/anime/3/815...,https://myanimelist.net/anime/31758/Kizumonoga...


In [9]:
myanime.drop(columns=['img_url','link'], inplace=True)

In [10]:
myanime

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,tag,title_normalized
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,Comedy Sports Drama School Shounen,"Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,haikyuu!! second season comedy sports drama sc...,haikyuu second season
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,Drama Music Romance School Shounen,"Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,shigatsu wa kimi no uso drama music romance sc...,shigatsu wa kimi no uso
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,Sci-Fi Adventure Mystery Drama Fantasy,"Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,made in abyss sci-fi adventure mystery drama f...,made in abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...",Action Military Adventure Comedy Drama Magic F...,"Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,fullmetal alchemist: brotherhood action milita...,fullmetal alchemist brotherhood
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,Action Mystery Supernatural Vampire,"Jan 6, 2017",1.0,214621,502,22.0,8.83,kizumonogatari iii: reiketsu-hen action myster...,kizumonogatari iii reiketsuhen
...,...,...,...,...,...,...,...,...,...,...,...,...
19306,32979,Flip Flappers,Cocona is an average middle schooler living wi...,Sci-Fi Adventure Comedy Magic,"Oct 6, 2016 to Dec 29, 2016",13.0,134252,843,1070.0,7.73,flip flappers sci-fi adventure comedy magic co...,flip flappers
19307,123,Fushigi Yuugi,"While visiting the National Library, junior-hi...",Adventure Fantasy Magic Martial Arts Comedy Ro...,"Apr 6, 1995 to Mar 28, 1996",52.0,84407,1292,1071.0,7.73,fushigi yuugi adventure fantasy magic martial ...,fushigi yuugi
19308,1281,Gakkou no Kaidan,"Years ago, all of the ghosts in a haunted scho...",Mystery Horror Supernatural,"Oct 22, 2000 to Mar 25, 2001",19.0,83093,1314,1073.0,7.73,gakkou no kaidan mystery horror supernatural y...,gakkou no kaidan
19309,450,InuYasha Movie 2: Kagami no Naka no Mugenjo,Inuyasha and company have finally destroyed Na...,Action Adventure Comedy Historical Demons Supe...,"Dec 21, 2002",1.0,71989,1469,1077.0,7.73,inuyasha movie 2: kagami no naka no mugenjo ac...,inuyasha movie 2 kagami no naka no mugenjo


In [11]:
myanime.to_pickle(
    processed_path / "anime.pkl"
)

In [12]:
myanime = pd.read_pickle(r"..\data\processed\anime.pkl")

In [13]:
myanime

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,tag,title_normalized
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,Comedy Sports Drama School Shounen,"Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,haikyuu!! second season comedy sports drama sc...,haikyuu second season
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,Drama Music Romance School Shounen,"Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,shigatsu wa kimi no uso drama music romance sc...,shigatsu wa kimi no uso
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,Sci-Fi Adventure Mystery Drama Fantasy,"Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,made in abyss sci-fi adventure mystery drama f...,made in abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...",Action Military Adventure Comedy Drama Magic F...,"Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,fullmetal alchemist: brotherhood action milita...,fullmetal alchemist brotherhood
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,Action Mystery Supernatural Vampire,"Jan 6, 2017",1.0,214621,502,22.0,8.83,kizumonogatari iii: reiketsu-hen action myster...,kizumonogatari iii reiketsuhen
...,...,...,...,...,...,...,...,...,...,...,...,...
19306,32979,Flip Flappers,Cocona is an average middle schooler living wi...,Sci-Fi Adventure Comedy Magic,"Oct 6, 2016 to Dec 29, 2016",13.0,134252,843,1070.0,7.73,flip flappers sci-fi adventure comedy magic co...,flip flappers
19307,123,Fushigi Yuugi,"While visiting the National Library, junior-hi...",Adventure Fantasy Magic Martial Arts Comedy Ro...,"Apr 6, 1995 to Mar 28, 1996",52.0,84407,1292,1071.0,7.73,fushigi yuugi adventure fantasy magic martial ...,fushigi yuugi
19308,1281,Gakkou no Kaidan,"Years ago, all of the ghosts in a haunted scho...",Mystery Horror Supernatural,"Oct 22, 2000 to Mar 25, 2001",19.0,83093,1314,1073.0,7.73,gakkou no kaidan mystery horror supernatural y...,gakkou no kaidan
19309,450,InuYasha Movie 2: Kagami no Naka no Mugenjo,Inuyasha and company have finally destroyed Na...,Action Adventure Comedy Historical Demons Supe...,"Dec 21, 2002",1.0,71989,1469,1077.0,7.73,inuyasha movie 2: kagami no naka no mugenjo ac...,inuyasha movie 2 kagami no naka no mugenjo


In [14]:
anime = pd.read_pickle(r"..\data\processed\anime.pkl")

In [15]:
anime.head()

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,tag,title_normalized
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,Comedy Sports Drama School Shounen,"Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,haikyuu!! second season comedy sports drama sc...,haikyuu second season
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,Drama Music Romance School Shounen,"Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,shigatsu wa kimi no uso drama music romance sc...,shigatsu wa kimi no uso
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,Sci-Fi Adventure Mystery Drama Fantasy,"Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,made in abyss sci-fi adventure mystery drama f...,made in abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...",Action Military Adventure Comedy Drama Magic F...,"Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,fullmetal alchemist: brotherhood action milita...,fullmetal alchemist brotherhood
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,Action Mystery Supernatural Vampire,"Jan 6, 2017",1.0,214621,502,22.0,8.83,kizumonogatari iii: reiketsu-hen action myster...,kizumonogatari iii reiketsuhen


In [16]:
import ast
import re

# Remove anime without synopsis
anime = anime.dropna(subset=["synopsis"]).copy()


def clean_genres(value):
    if pd.isna(value):
        return ""

    try:
        genres = ast.literal_eval(value)

        if isinstance(genres, list):
            return ", ".join(str(g).strip() for g in genres)

    except (ValueError, SyntaxError):
        pass

    return str(value)


def clean_text(value):
    if pd.isna(value):
        return ""

    value = str(value)

    # Remove newline characters
    value = re.sub(r"\s+", " ", value)

    # Remove the MAL rewrite marker
    value = re.sub(r"\[written by mal rewrite\]", "", value, flags=re.I)

    return value.strip()


anime["genre_clean"] = anime["genre"].apply(clean_genres)
anime["synopsis_clean"] = anime["synopsis"].apply(clean_text)

anime["tag2"] = (
    "Title: " + anime["title"].fillna("") +
    " Genres: " + anime["genre_clean"] +
    " Synopsis: " + anime["synopsis_clean"]
).str.lower()

In [17]:
anime['tag2'][0]

"title: haikyuu!! second season genres: comedy sports drama school shounen synopsis: following their participation at the inter-high, the karasuno high school volleyball team attempts to refocus their efforts, aiming to conquer the spring tournament instead. when they receive an invitation from long-standing rival nekoma high, karasuno agrees to take part in a large training camp alongside many notable volleyball teams in tokyo and even some national level players. by playing with some of the toughest teams in japan, they hope not only to sharpen their skills, but also come up with new attacks that would strengthen them. moreover, hinata and kageyama attempt to devise a more powerful weapon, one that could possibly break the sturdiest of blocks. facing what may be their last chance at victory before the senior players graduate, the members of karasuno's volleyball team must learn to settle their differences and train harder than ever if they hope to overcome formidable opponents old an

In [18]:
# Ch3cking  if tags will go beyond limit

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

token_counts = anime["tag2"].fillna("").apply(
    lambda x: len(tokenizer.encode(
        x,
        add_special_tokens=True,
        truncation=False
    ))
)

print("Total texts:", len(token_counts))
print("Texts > 256 tokens:", (token_counts > 256).sum())
print("Percentage > 256:", (token_counts > 256).mean() * 100)
print("Maximum tokens:", token_counts.max())
print("Average tokens:", token_counts.mean())

c:\Coding\Project Black Knight\project_black_knight\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (539 > 512). Running this sequence through the model will result in indexing errors


Total texts: 18336
Texts > 256 tokens: 939
Percentage > 256: 5.1210732984293195
Maximum tokens: 641
Average tokens: 113.76723385689354


In [19]:
print(token_counts[token_counts > 256].describe())

count    939.000000
mean     300.698616
std       51.401029
min      257.000000
25%      268.000000
50%      286.000000
75%      313.000000
max      641.000000
Name: tag2, dtype: float64


In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model max sequence length:", model.max_seq_length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 29394.66it/s]


Model max sequence length: 256


In [21]:
anime

,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,tag,title_normalized,genre_clean,synopsis_clean,tag2
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,Comedy Sports Drama School Shounen,"Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,haikyuu!! second season comedy sports drama sc...,haikyuu second season,Comedy Sports Drama School Shounen,Following their participation at the Inter-Hig...,title: haikyuu!! second season genres: comedy ...
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,Drama Music Romance School Shounen,"Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,shigatsu wa kimi no uso drama music romance sc...,shigatsu wa kimi no uso,Drama Music Romance School Shounen,Music accompanies the path of the human metron...,title: shigatsu wa kimi no uso genres: drama m...
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,Sci-Fi Adventure Mystery Drama Fantasy,"Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,made in abyss sci-fi adventure mystery drama f...,made in abyss,Sci-Fi Adventure Mystery Drama Fantasy,The Abyss—a gaping chasm stretching down into ...,title: made in abyss genres: sci-fi adventure ...
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...",Action Military Adventure Comedy Drama Magic F...,"Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,fullmetal alchemist: brotherhood action milita...,fullmetal alchemist brotherhood,Action Military Adventure Comedy Drama Magic F...,"""In order for something to be obtained, someth...",title: fullmetal alchemist: brotherhood genres...
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,Action Mystery Supernatural Vampire,"Jan 6, 2017",1.0,214621,502,22.0,8.83,kizumonogatari iii: reiketsu-hen action myster...,kizumonogatari iii reiketsuhen,Action Mystery Supernatural Vampire,After helping revive the legendary vampire Kis...,title: kizumonogatari iii: reiketsu-hen genres...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19306,32979,Flip Flappers,Cocona is an average middle schooler living wi...,Sci-Fi Adventure Comedy Magic,"Oct 6, 2016 to Dec 29, 2016",13.0,134252,843,1070.0,7.73,flip flappers sci-fi adventure comedy magic co...,flip flappers,Sci-Fi Adventure Comedy Magic,Cocona is an average middle schooler living wi...,title: flip flappers genres: sci-fi adventure ...
19307,123,Fushigi Yuugi,"While visiting the National Library, junior-hi...",Adventure Fantasy Magic Martial Arts Comedy Ro...,"Apr 6, 1995 to Mar 28, 1996",52.0,84407,1292,1071.0,7.73,fushigi yuugi adventure fantasy magic martial ...,fushigi yuugi,Adventure Fantasy Magic Martial Arts Comedy Ro...,"While visiting the National Library, junior-hi...",title: fushigi yuugi genres: adventure fantasy...
19308,1281,Gakkou no Kaidan,"Years ago, all of the ghosts in a haunted scho...",Mystery Horror Supernatural,"Oct 22, 2000 to Mar 25, 2001",19.0,83093,1314,1073.0,7.73,gakkou no kaidan mystery horror supernatural y...,gakkou no kaidan,Mystery Horror Supernatural,"Years ago, all of the ghosts in a haunted scho...",title: gakkou no kaidan genres: mystery horror...
19309,450,InuYasha Movie 2: Kagami no Naka no Mugenjo,Inuyasha and company have finally destroyed Na...,Action Adventure Comedy Historical Demons Supe...,"Dec 21, 2002",1.0,71989,1469,1077.0,7.73,inuyasha movie 2: kagami no naka no mugenjo ac...,inuyasha movie 2 kagami no naka no mugenjo,Action Adventure Comedy Historical Demons Supe...,Inuyasha and company have finally destroyed Na...,title: inuyasha movie 2: kagami no naka no mug...


In [22]:
from pathlib import Path
paths = Path(r"..\data\processed")
anime.to_pickle(paths/"anime_tag2.pkl")